# Country similarity pool ranking based on SINAS dataset

This notebook explores how to use the SINAS dataset (Goméz-Suárez et al., 2025) to create a horizon scanning priority species list. For it to run, please download the latest version of  the SINAS dataset from [this link](https://doi.org/10.5281/zenodo.18220953) and save it in the 'data' directory of the repository.

## 1. Setup

In [3]:
# Import dependencies
import pandas as pd
from pathlib import Path
import sys
import os

print("All dependencies imported successfully.")

All dependencies imported successfully.


## 2. Load in the SINAS dataset and preview file

In [ ]:
# Set file path
# Make sure our base path is set correctly for local src imports
sys.path.append(os.path.abspath('..'))
repo_root = Path.cwd().parent
file_path = repo_root / 'data' / 'SInAS_3.1.1.csv' #Check if version is up to date, if not, update to latest version

print(f"Loading data from: {file_path}") #verify if correct location

raw_data = pd.read_csv(
    file_path, 
    sep=None, 
    engine='python', #autodetect
    quotechar='"' #seperator
)

print("Data loaded successfully.")
print(f"Columns found: {raw_data.columns.tolist()}") #show column types

raw_data.head() #show df preview


Loading data from: c:\Users\simon\Documents\GitHub\horizon-scanner\data\SInAS_3.1.1.csv
Data loaded successfully.
Columns found: ['location', 'locationID', 'taxon', 'taxonID', 'eventDate', 'habitat', 'occurrenceStatus', 'establishmentMeans', 'degreeOfEstablishment', 'pathway', 'datasetName', 'bibliographicCitation']


,location,locationID,taxon,taxonID,eventDate,habitat,occurrenceStatus,establishmentMeans,degreeOfEstablishment,pathway,datasetName,bibliographicCitation
0,Aegean,101,Aphaenogaster splendida,5461,NaN,NaN,NaN,introduced,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."
1,Aegean,101,Camponotus fallax,5586,NaN,terrestrial,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."
2,Aegean,101,Camponotus vagus,5601,NaN,terrestrial,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."
3,Aegean,101,Cardiocondyla mauritanica,5609,NaN,terrestrial,NaN,introduced,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."
4,Aegean,101,Cataglyphis nodus,5620,NaN,NaN,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su..."


## 3. Verify if EU member states and surrounding areas are in location column 

In [3]:
# 1. Define the target countries
eu_27 = [
    'Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic', 
    'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 
    'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 
    'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 
    'Slovenia', 'Spain', 'Sweden'
]

uk = ['United Kingdom'] # You may need to add 'UK' or 'Great Britain' if your data uses those

# Broad list of European neighbors (EFTA, Balkans, Eastern Europe)
neighbors = [
    'Albania', 'Andorra', 'Belarus', 'Bosnia And Herzegovina', 'Iceland', 
    'Kosovo', 'Liechtenstein', 'Moldova', 'Monaco', 'Montenegro', 
    'North Macedonia', 'Norway', 'Russia', 'San Marino', 'Serbia', 
    'Switzerland', 'Turkey', 'Ukraine', 'Vatican City'
]

# Combine all lists into a single set for easy mathematical comparison
target_countries = set(eu_27 + uk + neighbors)

# 2. Extract unique locations from your dataset
# We drop NAs and convert to title case just in case there are lowercase entries
available_locations = set(
    raw_data['location']
    .dropna()
    .astype(str)
    .str.title()
    .unique()
    )

# 3. Calculate present and missing countries
present_countries = target_countries.intersection(available_locations)
missing_countries = target_countries.difference(available_locations)

# 4. Output the results
print("--- LOCATION CHECK RESULTS ---")
print(f"Total target countries: {len(target_countries)}")
print(f"Countries found in data: {len(present_countries)}")
print(f"Countries missing: {len(missing_countries)}\n")

if missing_countries:
    print("Missing Countries (Check for spelling/naming variations like 'Czechia' vs 'Czech Republic'):")
    for country in sorted(missing_countries):
        print(f" - {country}")
else:
    print("✅ All EU member states, the UK, and neighboring countries are present in the 'location' column!")

--- LOCATION CHECK RESULTS ---
Total target countries: 47
Countries found in data: 47
Countries missing: 0

✅ All EU member states, the UK, and neighboring countries are present in the 'location' column!


## 4. Enrich data with country centroids

In [4]:
from src.get_region_ranking import get_region_centroids #import function

# 1. Extract unique regions from the raw data
all_regions = set(raw_data['location'].dropna().astype(str).str.strip().str.title())

# 2. Get the coordinates dictionary
region_coords = get_region_centroids(list(all_regions))

# 3. Enrich the raw_data dataframe directly!
raw_data['location_clean'] = raw_data['location'].dropna().astype(str).str.strip().str.title()
raw_data['Centroid'] = raw_data['location_clean'].map(region_coords)

print("Raw data enriched with regional centroids!")
raw_data.head()

Fetching geographic centroids for 289 unique regions...


Geocoding Regions:   0%|          | 0/289 [00:00<?, ?it/s]

Raw data enriched with regional centroids!


,location,locationID,taxon,taxonID,eventDate,habitat,occurrenceStatus,establishmentMeans,degreeOfEstablishment,pathway,datasetName,bibliographicCitation,location_clean,Centroid
0,Aegean,101,Aphaenogaster splendida,5461,NaN,NaN,NaN,introduced,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",Aegean,"(38.0622276, 25.7205887)"
1,Aegean,101,Camponotus fallax,5586,NaN,terrestrial,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",Aegean,"(38.0622276, 25.7205887)"
2,Aegean,101,Camponotus vagus,5601,NaN,terrestrial,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",Aegean,"(38.0622276, 25.7205887)"
3,Aegean,101,Cardiocondyla mauritanica,5609,NaN,terrestrial,NaN,introduced,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",Aegean,"(38.0622276, 25.7205887)"
4,Aegean,101,Cataglyphis nodus,5620,NaN,NaN,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",Aegean,"(38.0622276, 25.7205887)"


In [3]:
#save raw_data as new file

# 1. Define the output path (e.g., in the same 'data' folder)
output_file_path = repo_root / 'data' / 'SInAS_enriched.csv'

# 2. Save the dataframe
# index=False prevents pandas from adding a new column for the row numbers
raw_data.to_csv(output_file_path, index=False)

print(f"File successfully saved to: {output_file_path}")


File successfully saved to: c:\Users\simon\Documents\GitHub\horizon-scanner\data\SInAS_enriched.csv


## 5. Enrich dataset with worldclim and worldbank data

In [5]:
from src.get_region_ranking import enrich_ias_data #import function

input_file_path = repo_root / 'data' / 'SInAS_enriched.csv'

enriched_data = pd.read_csv(
    input_file_path, 
    sep=None, 
    engine='python', #autodetect
    quotechar='"' #seperator
)

# 1. Run the enrichment on your initially enriched_data
# This filters for real countries, pulls economic, climate, and invasion stats
enriched_df = enrich_ias_data(enriched_data)

# 2. Cleanup: Drop intermediate API columns if needed
enriched_df = enriched_df.drop(columns=['economy', 'iso3'])

# 3. View the new features
print(f"Dataset enriched. New columns: {list(enriched_df.columns[-5:])}")
enriched_df.head()

Step 1: Validating countries...
Step 2: Fetching WDI (Economic) data for 288 countries...


Step 2/4: Fetching Economic Data:   0%|          | 0/29 [00:00<?, ?it/s]

Step 3: Fetching CCKP (Climate) and GBIF (Invasives) data...
 Processing Aegean...
 Processing AFG...
 Processing Akrotiri and Dhekelia...
 Processing ALA...
 Processing Alaska...
 Processing ALB...
 Processing DZA...
 Processing ASM...
 Processing Amsterdam Island...
 Processing AND...
 Processing AGO...
 Processing AIA...
 Processing ATA...
 Processing ATG...
 Processing Antipodes Island...
 Processing ARG...
 Processing ARM...
 Processing ABW...
 Processing Ascension...
 Processing AUS...
 Processing AUT...
 Processing AZE...
 Processing Azores...
 Processing BHS...
 Processing BHR...
 Processing Balearic Islands...
 Processing BGD...
 Processing BRB...
 Processing BLR...
 Processing BEL...
 Processing BLZ...
 Processing BEN...
 Processing BMU...
 Processing BTN...
 Processing BOL...
 Processing BES...
 Processing BIH...
 Processing BWA...
 Processing BVT...
 Processing BRA...
 Processing BRN...
 Processing BGR...
 Processing BFA...
 Processing BDI...
 Processing KHM...
 Processing 

,location,locationID,taxon,taxonID,eventDate,habitat,occurrenceStatus,establishmentMeans,degreeOfEstablishment,pathway,datasetName,bibliographicCitation,pop_density,annual_mean_temp,annual_precip,invasion_debt_count
0,Aegean,101,Aphaenogaster splendida,5461,NaN,NaN,NaN,introduced,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",NaN,None,None,0
1,Aegean,101,Camponotus fallax,5586,NaN,terrestrial,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",NaN,None,None,0
2,Aegean,101,Camponotus vagus,5601,NaN,terrestrial,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",NaN,None,None,0
3,Aegean,101,Cardiocondyla mauritanica,5609,NaN,terrestrial,NaN,introduced,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",NaN,None,None,0
4,Aegean,101,Cataglyphis nodus,5620,NaN,NaN,NaN,native,NaN,NaN,Ants_Wong,"Wong, M., Economo, E. P. & Guénard, B. Data su...",NaN,None,None,0


## 5. Get similarity in presence / absence between regions

In [5]:
from src.get_region_ranking import rank_regions_by_similarity #import function

#specify the target country

target_region = "Belgium"

#Query for rankings and similarity scores, comparing all species (not just introduced ones)

try:
    # Get the rankings (comparing all species)
    rankings = rank_regions_by_similarity(raw_data, target_region, compare_only_introduced=False)
    
    print(f"--- Top 100 Regions Most Similar to {target_region} ---")
    
    # Format the similarity score for a cleaner display
    display_df = rankings.head(100).copy()
    display_df['Similarity score'] = (display_df['Similarity score'] * 100).round(2).astype(str) + '%'
    
    display(display_df) # Use display() instead of print() in notebooks for a nicely formatted table

except ValueError as e:
    print(f"Error: {e}")

--- Top 100 Regions Most Similar to Belgium ---


,Region,Shared species,Total unique species in both,Target region total,Comparison region total,Similarity score
0,Netherlands,3274,6280,5007,4547,52.13%
1,United Kingdom,3125,7323,5007,5441,42.67%
2,Austria,2710,6589,5007,4292,41.13%
3,Switzerland,2415,5872,5007,3280,41.13%
4,Denmark,2575,6391,5007,3959,40.29%
...,...,...,...,...,...,...
95,Yemen,384,5874,5007,1251,6.54%
96,Israel,375,5952,5007,1320,6.3%
97,Peru,417,6805,5007,2215,6.13%
98,Tanzania,413,6907,5007,2313,5.98%


## 6. Get IAS risk probability score based on similarity and distance of other regions

In [8]:
from src.get_region_ranking import predict_ias_risk, rank_regions_by_similarity #import function

#set target region for similarity rankging and risk prediction
target_region_name = "Belgium" #can adapt target country/region

# 1. First, get the similarity scores (# species overlap) for the target region. 
# (We compare against all species to get a true ecological baseline)
similarity_rankings = rank_regions_by_similarity(
    raw_data, 
    target_region_name, 
    compare_only_introduced=False #change to True if you want to compare only based on introduced species
)

# 2. Pass that similarity dataframe into the risk predictor
ias_risk_predictions = predict_ias_risk(
    df=raw_data, 
    similarity_df=similarity_rankings, 
    target_region="Belgium", #can adapt target country/region 
    species_to_validate=100 #can adapt length of final horizon scan list
)


Ranking species locally using a Hybrid Cumulative Spatial Score...
Validating top threats against GBIF for 'BE'...


True Threats Found:   0%|          | 0/100 [00:00<?, ?species/s]

In [11]:
# Display the top the species output with formatted scores for easier reading
print(f"--- Top IAS Threats for {target_region_name} ---")

display_df = ias_risk_predictions.copy()
display_df['Hybrid Risk Score'] = display_df['Hybrid Risk Score'].round(4)
display_df['Max Single-Region Similarity'] = (display_df['Max Single-Region Similarity'] * 100).round(2).astype(str) + '%'

display(display_df)

--- Top IAS Threats for Belgium ---


,Species,Hybrid Risk Score,Max Single-Region Similarity,Found In Regions,Region Count
0,Plum pox virus,5.0961,52.13%,"Albania, Argentina, Austria, Azores, Belarus, ...",53
1,Cuscuta pentagona,4.7572,52.13%,"Albania, Algeria, Argentina, Armenia, Banglade...",90
2,"Tuta absoluta (Meyrick, Tuta absoluta 1917)",3.7454,52.13%,"Albania, Austria, Balearic Islands, Bulgaria, ...",28
3,Chenopodium ambrosioides,3.5311,52.13%,"Albania, Angola, Anguilla, Australia, Austria,...",51
4,Diaspidiotus perniciosus,3.2942,52.13%,"Australia, Austria, Bulgaria, China, Corsica, ...",26
...,...,...,...,...,...
95,Schyzocotyle acheilognathi,1.5998,42.67%,"Austria, France, Germany, Hawaii, Honduras, Me...",13
96,Sorbus intermedia,1.5955,42.67%,"Armenia, Austria, Canada, Czech Republic, Germ...",12
97,Aphis forbesi,1.5945,41.13%,"Austria, Bulgaria, Czech Republic, France, Ger...",9
98,Illinoia liriodendri,1.5941,52.13%,"Corsica, Czech Republic, France, Italy, Japan,...",9
